# Estadística Descriptiva e Inferencial## Módulo 1 · Clase 1 — Del dato a la variable aleatoria**Notebook 01** · Duración: 60 minutos · **No necesitas saber programar para seguir esto**---### Cómo se usa este notebookEste documento tiene dos tipos de bloques:- **Bloques de texto** (como este): explican qué vamos a hacer y por qué.- **Bloques de código** (los que tienen fondo distinto): son las instrucciones que la computadora  ejecuta. Para ejecutar uno, haz clic dentro y presiona **`Shift` + `Enter`** al mismo tiempo.**Reglas de la casa:**1. Ejecuta las celdas **en orden, de arriba hacia abajo**. Si te saltas una, las siguientes fallan.2. Todo lo que empieza con `#` es un **comentario**: la computadora lo ignora, está ahí para ti.3. Si una celda da error, no te asustes. Lo más común es haberse salteado una celda anterior.4. Las celdas que dicen `# TODO` las completas tú. Las demás solo se ejecutan.### Qué vas a poder hacer al terminar1. Demostrar con datos por qué el promedio, solo, es un número peligroso.2. Ver con tus propios ojos cómo mejora una estimación cuando tienes más datos.3. Programar una fórmula de probabilidad desde cero y comprobar que está bien.4. Descubrir cuándo una fórmula **no** sirve para tus datos (esto es lo más valioso).5. Repetir dos análisis históricos reales: uno de 1898 y uno de 1944.

---### Recordatorio de símbolosCada vez que aparezca uno de estos en el código o en el texto, significa esto:| Símbolo | Se lee | Qué es ||---|---|---|| μ | "mu" | El promedio real de toda la población (no lo conoces) || x̄ | "x barra" | El promedio que tú calculaste con tus datos || σ | "sigma" | La dispersión real de la población || s | "ese" | La dispersión que tú calculaste || σ² , s² | "sigma / ese cuadrado" | La dispersión elevada al cuadrado (la varianza) || p | "pe" | Una probabilidad. Va de 0 a 1: `p = 0.12` es 12% || p̂ | "pe gorro" | La probabilidad que tú calculaste. En el código la escribimos `p_hat` || n | "ene" | Cuántos datos tienes || k | "ka" | Un valor concreto de la variable || λ | "lambda" | Una tasa: cuántas cosas pasan por hora. En el código: `lam` || e | "e" | Un número fijo, 2.71828… Igual que π, ya existe || k! | "ka factorial" | Multiplicar en bajada: 4! = 4×3×2×1 = 24 |> El **sombrerito** (`^`) siempre quiere decir lo mismo: *"esto es mi estimación, no la verdad"*.> Como la etiqueta "escala aproximada" de un mapa: el mapa sirve, pero te avisa que no es el terreno.

---## 01 · Preparar todo### ¿Qué es una librería?Una **librería** es una caja de herramientas que alguien más ya programó. En lugar de escribir lafórmula de la desviación estándar cada vez, la pides prestada. Vamos a usar cuatro:| Librería | Para qué sirve | Cómo la llamamos en el código ||---|---|---|| `numpy` | Hacer matemáticas con listas largas de números, rápido | `np` || `pandas` | Manejar tablas, como una hoja de Excel dentro de Python | `pd` || `matplotlib` | Dibujar gráficos | `plt` || `scipy.stats` | Las distribuciones ya programadas: Bernoulli, Binomial, Poisson… | `stats` |En Colab ya están instaladas: no hay que descargar nada. Solo hay que "abrir la caja", que es loque hace la palabra `import`.

In [ ]:
# "import X as Y" significa: trae la librería X y de ahora en adelante le digo Y (para escribir menos)import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom scipy import stats          # "from A import B" = de la caja A saca solo la herramienta B# ── La semilla ────────────────────────────────────────────────────────────────# Vamos a generar números al azar. Si no fijamos una "semilla", cada vez que ejecutes# esto obtendrás números distintos y tus resultados no coincidirán con los míos.# La semilla es como decirle a la computadora: "usa SIEMPRE este mismo azar".SEED = 42                                  # 42 es una convención, podría ser cualquier númerorng = np.random.default_rng(SEED)          # rng = generador de números al azar ("random number generator")# ── Estilo de los gráficos (puramente cosmético, no afecta los cálculos) ──────plt.rcParams.update({    "figure.figsize": (9, 4.5),            # ancho y alto de cada gráfico, en pulgadas    "axes.grid": True,                     # dibuja la cuadrícula de fondo    "grid.alpha": 0.25,                    # qué tan tenue es esa cuadrícula (0 = invisible, 1 = sólida)    "axes.spines.top": False,              # quita el borde de arriba    "axes.spines.right": False,            # quita el borde de la derecha    "font.size": 11,})# Nuestros tres colores, guardados en variables para no repetir los códigosAZUL, ROSA, NAVY = "#1A56E8", "#E6115E", "#0A2559"# print() sirve para mostrar algo en pantallaprint("Todo listo. Versión de numpy:", np.__version__)

### Los datos con los que vamos a trabajarNo vamos a descargar nada: los vamos a **inventar**. Y eso no es hacer trampa, es una técnica.Cuando inventas los datos tú, **conoces la respuesta verdadera de antemano**. Entonces puedescomprobar si tu método la encuentra. Con datos reales nunca sabes la respuesta verdadera, así queno puedes saber si tu método funciona o te está mintiendo.Vamos a simular una semana de una web:| Columna | Qué guarda ||---|---|| `ingreso` | Cuánto gana al mes cada usuario, en soles || `convirtio` | 1 si el usuario compró, 0 si no compró |

In [ ]:
N = 567                        # N = cuántos usuarios vamos a simular# ── Columna 1: el ingreso ────────────────────────────────────────────────────# "lognormal" es una forma de generar números torcidos hacia la derecha:# muchos valores medianos y unos pocos altísimos. Así se comporta casi todo el dinero del mundo.# mean y sigma son los controles de esa forma; no hace falta entenderlos hoy.ingreso = rng.lognormal(mean=np.log(2750), sigma=0.62, size=N).round(0)#                                                       ^^^^^^^ .round(0) = sin decimales# ── Columna 2: ¿compró o no? ─────────────────────────────────────────────────# Esto es una Bernoulli: cada usuario compra (1) o no compra (0).P_VERDADERO = 0.12             # el 12% compra. NOSOTROS lo definimos, así que lo sabemos.convirtio = rng.binomial(1, P_VERDADERO, size=N)   # el "1" significa: un solo intento por persona# ── Armamos la tabla ─────────────────────────────────────────────────────────# Un DataFrame es una tabla, igual que una hoja de Excel. Se define con { "nombre": datos }usuarios = pd.DataFrame({"ingreso": ingreso, "convirtio": convirtio})usuarios.head()                # .head() muestra solo las primeras 5 filas, para no llenar la pantalla

In [ ]:
# .describe() calcula de golpe un resumen de cada columna.# La .T al final voltea la tabla (transpone) para que se lea más cómodo.usuarios.describe().T

En ese resumen ya aparecen palabras conocidas:- `count` = cuántos datos hay (nuestro **n**)- `mean` = el promedio (nuestro **x̄**)- `std` = la desviación estándar (nuestro **s**), del inglés *standard deviation*- `50%` = la **mediana**, el valor que queda al medio- `min` y `max` = el más chico y el más grande

---## 02 · El promedio mienteEn clase lo hicimos a mano con cinco números. Ahora con 567, y con la computadora.> **Analogía — el río de 1.20 m.** Un río con profundidad promedio de 1.20 m es un dato correcto,> y te puedes ahogar cruzándolo: el promedio no te avisa del pozo de 3 metros que hay en el medio.

In [ ]:
x = usuarios["ingreso"]        # los corchetes con un nombre adentro = "dame esta columna"# Cada línea calcula una cosa y la guarda en una variablemedia   = x.mean()                                  # el promedio (x̄)mediana = x.median()                                # el valor del mediosd      = x.std(ddof=1)                             # la desviación estándar (s)#                ^^^^^^ ddof=1 le dice: divide entre n−1, no entre n.#                       Es exactamente el "¿por qué entre 4 y no entre 5?" de la clase.cv      = sd / media                                # coeficiente de variacióniqr     = x.quantile(0.75) - x.quantile(0.25)       # rango entre cuartiles# Mostramos todo. Lo de {media:>9,.0f} solo controla la alineación y los decimales.print(f"Promedio (x̄)        : S/ {media:>9,.0f}")print(f"Mediana             : S/ {mediana:>9,.0f}")print(f"Desviación est. (s) : S/ {sd:>9,.0f}")print(f"Coef. de variación  : {cv:>12.2f}")print(f"Rango entre cuartiles: S/ {iqr:>8,.0f}")print()print(f"El promedio es {100*(media/mediana - 1):.1f}% más alto que la mediana.")

### Veámoslo dibujadoUn **histograma** es un gráfico que cuenta cuántos datos caen en cada rango. Las barras altas sonlos rangos donde hay mucha gente.

In [ ]:
fig, ax = plt.subplots()       # crea un lienzo vacío (fig) y unos ejes para dibujar (ax)ax.hist(x, bins=45, color=AZUL, alpha=0.75, edgecolor="white")#          ^^^^^^^ en cuántas barras partimos el rango#                            ^^^^^^^^^^ alpha = transparencia (0.75 = un poco translúcido)# axvline dibuja una línea vertical. lw = grosor, ls = estilo ("--" es punteada)ax.axvline(media,   color=ROSA, lw=2.5, label=f"Promedio = S/ {media:,.0f}")ax.axvline(mediana, color=NAVY, lw=2.5, ls="--", label=f"Mediana = S/ {mediana:,.0f}")ax.set_title("Ingreso mensual: el promedio queda a la derecha del usuario típico")ax.set_xlabel("Ingreso en soles")ax.set_ylabel("Cantidad de usuarios")ax.legend()                    # muestra el cuadrito con las etiquetasplt.show()                     # y finalmente lo dibuja en pantalla

Fíjate en la forma: la mayoría de la gente está a la izquierda, y hay una **cola larga** hacia laderecha con unos pocos que ganan mucho. Esa cola es la que jala el promedio.### El experimento del dato extremoVamos a agregar **una sola persona** que gana 900,000 soles al mes, y a mirar qué se mueve.

In [ ]:
# pd.concat pega dos conjuntos de datos uno debajo del otro.# pd.Series([900_000]) es una "columna" con un solo valor. El guion bajo en 900_000# es solo para leerlo mejor: Python lo entiende como 900000.x_sucio = pd.concat([x, pd.Series([900_000])], ignore_index=True)# Armamos una tabla comparativa a manocomparacion = pd.DataFrame({    "sin el dato extremo": [x.mean(), x.median(), x.std(ddof=1),                            x.quantile(.75) - x.quantile(.25)],    "con el dato extremo": [x_sucio.mean(), x_sucio.median(), x_sucio.std(ddof=1),                            x_sucio.quantile(.75) - x_sucio.quantile(.25)],}, index=["promedio", "mediana", "desviación", "rango cuartiles"])#    ^^^^^ index = los nombres de las filas# Agregamos una columna calculada: cuánto cambió, en porcentajecomparacion["cambió en %"] = (comparacion["con el dato extremo"] /                              comparacion["sin el dato extremo"] - 1) * 100comparacion.round(1)           # .round(1) = un solo decimal, para que se lea limpio

> **Mira la última columna y deténte aquí un momento.**>> Un registro entre 568 movió el promedio y la desviación. **No tocó la mediana ni el rango entre> cuartiles.** A eso se le dice que la mediana es *robusta*: no se deja empujar por un dato loco.>> Y ahora la pregunta de trabajo: si tuvieras que fijar el **límite de una tarjeta de crédito**> para esta cartera de clientes, ¿lo calcularías con el promedio o con la mediana? ¿Y qué le> responderías al gerente comercial que quiere usar el promedio *porque es más alto*?

### Ejercicio 2.1El **promedio recortado** consiste en tirar el 5% más bajo y el 5% más alto, y promediar el resto.

In [ ]:
# TODO: calcula el promedio recortado al 5% de la columna de ingresos.# La herramienta ya existe: stats.trim_mean(datos, cuánto_recortar)# "cuánto_recortar" se escribe como decimal: 5% se escribe 0.05promedio_recortado = ...     # <- reemplaza los tres puntos por tu código# Cuando lo tengas, borra el # de la línea siguiente para ver el resultado:# print(f"Promedio recortado al 5%: S/ {promedio_recortado:,.0f}")# Y responde aquí, en un comentario:# ¿Quedó más cerca del promedio normal o de la mediana? ¿Por qué crees que pasa eso?

---## 03 · Con más datos, mejor estimaciónLa columna `convirtio` es una **Bernoulli**: cada usuario compra (1) o no compra (0).Nosotros sabemos que el `p` verdadero es 0.12 porque lo escribimos nosotros. En la vida real**jamás** lo sabes: solo tienes `p̂` (`p_hat` en el código), tu estimación. La pregunta que decidetodo es: *¿cuántos datos necesito para que mi estimación sirva de algo?*> **Analogía — el casino.** La casa no gana cada mano; gana con miles de manos. `p̂` se parece a `p`> cuando hay volumen, no cuando hay una tarde de suerte. Un reporte de conversión con 40 usuarios> es una tarde de suerte.

In [ ]:
# La conversión estimada es simplemente el promedio de la columna de ceros y unos.# ¿Por qué? Porque si de 10 personas compraron 2, la columna es [0,0,1,0,0,1,0,0,0,0]# y su promedio es 2/10 = 0.2, que es justamente la proporción que compró.p_hat = usuarios["convirtio"].mean()var_hat = p_hat * (1 - p_hat)              # la varianza de una Bernoulli: p(1−p)error_estandar = np.sqrt(var_hat / N)      # np.sqrt = raíz cuadrada ("square root")print(f"p verdadero (lo sabemos porque lo inventamos) : {P_VERDADERO:.4f}")print(f"p̂  (lo que estimamos con n = {N})              : {p_hat:.4f}")print(f"Varianza p(1−p)                                : {var_hat:.4f}")print(f"Error estándar de p̂                            : {error_estandar:.4f}")print()print("El error estándar es 'cuánto suele equivocarse mi estimación'.")print(f"O sea: con {N} usuarios, mi p̂ típicamente falla por ±{error_estandar:.4f}, es decir ±{error_estandar*100:.2f} puntos porcentuales.")

### La convergencia, dibujadaVamos a simular 20,000 usuarios llegando uno por uno, y a mirar cómo se mueve `p̂` a medida quellegan.

In [ ]:
n_max = 20_000muestras = rng.binomial(1, P_VERDADERO, size=n_max)   # 20,000 ceros y unos# np.cumsum va sumando acumulado: de [0,1,1,0] hace [0,1,2,2]# np.arange(1, n+1) genera [1,2,3,...,n]# Al dividir uno entre otro obtenemos el promedio hasta cada punto: eso es p̂ paso a paso.p_acumulado = np.cumsum(muestras) / np.arange(1, n_max + 1)fig, ax = plt.subplots()ax.plot(np.arange(1, n_max + 1), p_acumulado, color=AZUL, lw=1.2)ax.axhline(P_VERDADERO, color=ROSA, lw=2, ls="--", label=f"p verdadero = {P_VERDADERO}")#  ^^^^^^^^ axhline = línea HORIZONTAL (la vertical era axvline)ax.set_xscale("log")           # escala logarítmica: comprime el eje X para ver bien el inicioax.set_ylim(0, 0.35)           # limita el eje Y de 0 a 0.35 para que no se vea aplastadoax.set_title("Cómo p̂ se va acercando a p (ojo: no es rápido)")ax.set_xlabel("Usuarios acumulados (escala logarítmica)")ax.set_ylabel("p̂ estimado hasta ese punto")ax.legend()plt.show()# Y ahora los números concretos en algunos puntosprint("n         p̂        error")for n in [10, 50, 100, 500, 1_000, 5_000, 20_000]:    est = p_acumulado[n - 1]                          # [n-1] porque Python cuenta desde 0    print(f"{n:>6,}   {est:.4f}   {abs(est - P_VERDADERO):+.4f}")

### La consecuencia más importante de toda la clase`Var(X) = p(1−p)` es **más grande cuando p vale 0.5** y se hace chiquita en los extremos. Pero elerror *comparado con el tamaño de lo que mides* hace lo contrario: medir algo que pasa el 1% delas veces es carísimo.

In [ ]:
ps = np.array([0.01, 0.03, 0.05, 0.12, 0.30, 0.50])   # varias tasas de conversión posiblesn_ref = 1_000                                          # supongamos siempre 1,000 usuariostabla = pd.DataFrame({    "p (la tasa real)": ps,    "Varianza p(1−p)": (ps * (1 - ps)).round(4),    "Error estándar": np.sqrt(ps * (1 - ps) / n_ref).round(4),})# El error RELATIVO es el error comparado con el tamaño de p. Es el que importa en la práctica.tabla["Error relativo %"] = (tabla["Error estándar"] / tabla["p (la tasa real)"] * 100).round(1)tabla

> **Lee solo la última columna.**>> Con 1,000 usuarios, una conversión del **1%** se estima con un error relativo de ~31%: o sea que> prácticamente no la estás midiendo. Una del **50%** se estima con ~3%.>> Esta tabla es la razón por la que una prueba A/B sobre algo que pasa poco necesita **meses** de> tráfico. Volvemos a esto en la Clase 12 para calcular el tamaño de muestra con una fórmula.

### Ejercicio 3.1

In [ ]:
# TODO: repite el experimento de la convergencia, pero con p = 0.01 (un evento raro).# Pasos sugeridos:#   1. genera las muestras con rng.binomial(1, 0.01, size=n_max)#   2. calcula p_acumulado igual que arriba, con np.cumsum#   3. imprime el error en n = 100, 1000, 10000 y 20000# Pregunta a responder: ¿a partir de qué n el error se vuelve razonable?...

---## 04 · PMF y CDF: programando la fórmula**Regla de la casa:** una fórmula que no puedes programar, no la entiendes de verdad.Vamos a escribir la Binomial con nuestras propias manos y después comprobar que da lo mismo quela versión oficial de `scipy`. La fórmula, la misma de la slide 7:$$P(X = k) = \binom{n}{k}\, p^{k} (1-p)^{n-k}$$Donde:- $\binom{n}{k}$ (que en el código escribimos `comb(n, k)`) = de cuántas formas puede pasar- $p^{k}$ = la chance de que los k que sí, sí- $(1-p)^{n-k}$ = la chance de que los otros n−k fallen> **Analogía — velocímetro y odómetro.** La PMF es el velocímetro: el valor exacto en este punto.> La CDF es el odómetro: todo lo acumulado hasta aquí, y nunca baja.

In [ ]:
from math import comb          # comb(n, k) calcula las combinaciones. Ya viene con Python.# ── Definimos nuestra propia función ─────────────────────────────────────────# "def" significa "define una función". Una función es una receta a la que le pasas# ingredientes (k, n, p) y te devuelve un resultado.def binomial_pmf(k, n, p):    '''Calcula P(X = k) para una Binomial(n, p), tal cual la fórmula.'''    return comb(n, k) * p**k * (1 - p)**(n - k)    #      ^^^^^^^^^^   ^^^^^   ^^^^^^^^^^^^^^^^    #      pedazo 1     pedazo 2      pedazo 3      (** significa "elevado a")n, p = 30, 0.30                # 30 intentos, 30% de chance cada unoks = np.arange(0, n + 1)       # ks = [0, 1, 2, ..., 30], todos los resultados posibles# Calculamos la PMF de dos maneras y comparamospmf_mano  = np.array([binomial_pmf(k, n, p) for k in ks])   # con NUESTRA funciónpmf_scipy = stats.binom.pmf(ks, n, p)                        # con la versión oficial# np.allclose pregunta: "¿son iguales estos dos conjuntos de números?"print("¿Nuestra fórmula da lo mismo que scipy?", np.allclose(pmf_mano, pmf_scipy))print("Suma de todas las probabilidades:", pmf_mano.sum().round(10), " <- tiene que ser 1")

Que la suma dé exactamente 1 no es un detalle: es tu **primer chequeo siempre**. Si no da 1,hay un error en algún lado.

In [ ]:
cdf_scipy = stats.binom.cdf(ks, n, p)      # la acumulada, ya lista en scipy# plt.subplots(1, 2) crea dos gráficos lado a lado. axes[0] es el izquierdo, axes[1] el derecho.fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))axes[0].bar(ks, pmf_scipy, color=AZUL, alpha=0.85, edgecolor="white")axes[0].axvline(n * p, color=ROSA, ls="--", lw=2, label=f"E[X] = n×p = {n*p:.0f}")axes[0].set_title(f"PMF · P(X = k) · Binomial(n={n}, p={p})")axes[0].set_xlabel("k (cuántos éxitos)")axes[0].legend()# .step dibuja escalones en vez de una línea suave. Es lo correcto para una CDF discreta:# la probabilidad salta de golpe en cada valor entero, no sube en diagonal.axes[1].step(ks, cdf_scipy, where="post", color=ROSA, lw=2.5)axes[1].set_title("CDF · P(X ≤ k) · acumulada")axes[1].set_xlabel("k")axes[1].set_ylim(0, 1.05)plt.tight_layout()             # acomoda los dos gráficos para que no se pisenplt.show()

### El error del ±1: el bug número unoQuieres saber `P(X ≥ 12)`, o sea "12 o más". La tentación es escribir `1 - cdf(12)`.**Está mal.** Y la razón es simple: `cdf(12)` ya incluye al 12. Si le restas eso a 1, estásquitando también al 12, y el 12 **sí** cuenta. Lo correcto es `1 - cdf(11)`.Vamos a comprobarlo de cuatro formas distintas.

In [ ]:
k0 = 12mal     = 1 - stats.binom.cdf(k0, n, p)        # esto en realidad es P(X ≥ 13)bien    = 1 - stats.binom.cdf(k0 - 1, n, p)    # esto sí es P(X ≥ 12)directo = stats.binom.sf(k0 - 1, n, p)         # sf = "survival function" = 1 − cdf, ya hechafuerza  = pmf_scipy[ks >= k0].sum()            # sumar a mano todas las probabilidades de 12 en adelanteprint(f"1 − cdf(12)             = {mal:.6f}   <- MAL (esto es P(X ≥ 13))")print(f"1 − cdf(11)             = {bien:.6f}   <- BIEN")print(f"sf(11)                  = {directo:.6f}   <- lo mismo, más corto")print(f"suma directa desde k=12 = {fuerza:.6f}   <- la verdad de referencia")print()print(f"La diferencia entre hacerlo mal y bien es de {abs(bien-mal)*100:.1f} puntos porcentuales.")print("En un informe, eso es la diferencia entre aprobar y no aprobar un proyecto.")

### Ejercicio 4.1 — un caso de negocio completo

In [ ]:
# EL CONTEXTO# Esperas 1,000 visitantes este mes. Tu conversión histórica es del 3%.# La meta que te puso el jefe comercial es cerrar 40 ventas o más.n_visitas, p_conv, meta = 1_000, 0.03, 40# TODO 1: ¿cuántas ventas esperas en promedio, y cuánto suelen variar?#         Pistas: E[X] = n × p        y        desviación = raíz cuadrada de n × p × (1−p)#         Para la raíz usa np.sqrt(...)esperadas = ...desviacion = ...# TODO 2: ¿cuál es la probabilidad de alcanzar la meta, es decir P(X ≥ 40)?#         Cuidado con el ±1 que acabamos de ver. Usa stats.binom.sf(...)prob_meta = ...# TODO 3: ¿cuántas ventas superas en el 90% de los meses?#         Pista: stats.binom.ppf(0.10, n, p) — ppf es la CDF al revés:#         le das una probabilidad y te devuelve el valor de k que le corresponde.piso_90 = ...# Cuando termines, borra los # de estas líneas:# print(f"Esperadas: {esperadas:.1f} ventas, con variación típica de ±{desviacion:.1f}")# print(f"Probabilidad de llegar a {meta}: {prob_meta:.2%}")# print(f"En 9 de cada 10 meses cierras al menos {piso_90:.0f} ventas")

---## 05 · Poisson: cuándo sirve y cuándo noPoisson tiene una firma inconfundible: **el promedio y la varianza son el mismo número, λ**.Eso la hace fácil de usar y, sobre todo, fácil de **refutar**: si en tus datos la varianza esmucho más grande que el promedio, Poisson es el modelo equivocado. A eso se le llama**sobredispersión** (los datos están más desparramados de lo que la fórmula permite).> **Analogía — gotas de lluvia sobre una loseta.** No sabes cuál gota ni cuándo, pero sí cuántas> por minuto. Y si la lluvia viene en ráfagas en lugar de caer parejo, la tasa dejó de ser> constante: eso es sobredispersión.Para detectarlo hay un chequeo de una sola línea: el **índice de dispersión**, que es`varianza ÷ promedio`. Si vale cerca de 1, Poisson es plausible. Si vale 4 o 5, no.

In [ ]:
# ── CASO A: un call center normal, donde la tasa sí es constante ──────────────LAMBDA_REAL = 3.5              # 3.5 reclamos por hora, en promediohoras = 720                    # un mes de operación (30 días × 24 horas)reclamos = rng.poisson(LAMBDA_REAL, size=horas)# La mejor estimación de λ es, simplemente, el promedio de los datos.lambda_hat = reclamos.mean()varianza   = reclamos.var(ddof=1)print(f"λ estimado (el promedio) : {lambda_hat:.3f}")print(f"Varianza observada       : {varianza:.3f}")print(f"Índice de dispersión     : {varianza/lambda_hat:.3f}   <- cerca de 1 = Poisson sirve")

In [ ]:
# Definimos una función para no repetir el mismo gráfico dos vecesdef comparar_poisson(datos, titulo, ax):    '''Dibuja lo observado contra lo que predice Poisson, y muestra el índice de dispersión.'''    kmax = datos.max()    ks = np.arange(0, kmax + 1)    # (datos == k).mean() cuenta qué proporción de los datos vale exactamente k    obs = np.array([(datos == k).mean() for k in ks])    esp = stats.poisson.pmf(ks, datos.mean())        # lo que Poisson predeciría    ax.bar(ks - 0.2, obs, width=0.4, color=AZUL, label="Lo que pasó")    ax.bar(ks + 0.2, esp, width=0.4, color=ROSA, label="Lo que predice Poisson")    #      ^^^^^^^^ el −0.2 y +0.2 corren las barras a los lados para que no se tapen    indice = datos.var(ddof=1) / datos.mean()    ax.set_title(f"{titulo}\npromedio={datos.mean():.2f} · varianza={datos.var(ddof=1):.2f} · índice={indice:.2f}")    ax.set_xlabel("reclamos en una hora")    ax.set_ylabel("proporción de horas")    ax.legend()fig, ax = plt.subplots()comparar_poisson(reclamos, "CASO A — tasa constante", ax)plt.tight_layout(); plt.show()

### Caso B: el mismo promedio, pero el mundo realEn la práctica la tasa **no es constante**. De madrugada entran 1.2 reclamos por hora; en horapunta entran 9. Vamos a mezclar los dos horarios y a ajustar **una sola** Poisson encima, que esjusto lo que haría un reporte descuidado.

In [ ]:
# np.concatenate pega dos conjuntos de datos en uno solomezcla = np.concatenate([    rng.poisson(1.2, size=480),    # 480 horas tranquilas    rng.poisson(9.0, size=240),    # 240 horas de hora punta])fig, ax = plt.subplots()comparar_poisson(mezcla, "CASO B — dos horarios mezclados", ax)plt.tight_layout(); plt.show()print(f"Promedio del caso A: {reclamos.mean():.2f}   ·   índice: {reclamos.var(ddof=1)/reclamos.mean():.2f}")print(f"Promedio del caso B: {mezcla.mean():.2f}   ·   índice: {mezcla.var(ddof=1)/mezcla.mean():.2f}   <- muy por encima de 1")

> **Esto es lo más importante del notebook.**>> Los dos casos tienen un promedio parecido. Así que un reporte que diga *"recibimos unos 3.5> reclamos por hora"* describe igual de bien dos operaciones **completamente distintas**. El caso B> necesita el doble de agentes en hora punta; el caso A no necesita nada.>> El índice de dispersión es tu chequeo de una línea antes de creerle a un promedio. Si está muy> por encima de 1: **segmenta** (separa los horarios y analiza cada uno).

### Ejercicio 5.1 — la prueba formalEsto es un adelanto de la **Clase 10**. Ejecútalo y mira la conclusión; la teoría la construimosdespués. Lo único que hace la prueba es comparar, número por número, lo observado contra loesperado, y decir si la diferencia es demasiado grande para ser casualidad.

In [ ]:
def chi2_poisson(datos, kmax=None):    '''Compara lo observado con lo esperado bajo Poisson y devuelve un p-valor.'''    lam = datos.mean()    kmax = kmax or int(datos.max())    ks = np.arange(0, kmax)    # obs = cuántas veces salió cada valor. El último grupo junta "kmax o más".    obs = np.array([(datos == k).sum() for k in ks] + [(datos >= kmax).sum()], dtype=float)    esp = np.append(stats.poisson.pmf(ks, lam), stats.poisson.sf(kmax - 1, lam)) * len(datos)    # La prueba exige que cada grupo tenga al menos 5 casos esperados.    # Si el último tiene menos, lo juntamos con el anterior. Se repite hasta cumplir.    while len(esp) > 2 and esp[-1] < 5:        obs[-2] += obs[-1]; esp[-2] += esp[-1]        obs, esp = obs[:-1], esp[:-1]    chi2 = ((obs - esp) ** 2 / esp).sum()      # la fórmula del estadístico chi-cuadrado    gl = len(obs) - 1 - 1                      # "grados de libertad": −1 por estimar λ    p_valor = stats.chi2.sf(chi2, gl)    return chi2, gl, p_valorfor nombre, datos in [("Caso A (tasa constante)", reclamos), ("Caso B (mezclado)", mezcla)]:    chi2, gl, pv = chi2_poisson(datos)    veredicto = "Poisson sirve" if pv > 0.05 else "Poisson NO sirve"    print(f"{nombre:<26} chi2={chi2:>8.1f}  p={pv:.2e}  ->  {veredicto}")print()print("Por ahora quédate solo con esto: un p chico (menos de 0.05) significa")print("'la diferencia es demasiado grande para ser casualidad'.")

---## 06 · Los casos históricos, con datos de verdadHasta aquí ajustamos Poisson a datos que nosotros mismos inventamos. Cómodo, pero un poco tramposo.Ahora los dos conjuntos de datos que hicieron famosa a esta fórmula. Son los números originales,publicados en **1898** y en **1946**.

### 6.1 · Bortkiewicz (1898) — muertes por patada de caballoLadislaus von Bortkiewicz recopiló las muertes por patada de caballo en **10 cuerpos de caballeríadel ejército prusiano durante 20 años**.Primero una aclaración de vocabulario: un **"cuerpo-año"** es un cuerpo de caballería observadodurante un año. 10 cuerpos × 20 años = **200 cuerpos-año**. En total hubo 122 muertes.Lo notable del caso es que los cuerpos **no eran iguales entre sí** (distinto tamaño, distintaorganización) y aun así el patrón junto sale Poisson. Bortkiewicz lo llamó *la ley de losnúmeros pequeños*.

In [ ]:
# Los datos originales, tal como los publicó en 1898.# Se leen así: hubo 109 cuerpos-año con 0 muertes, 65 con 1 muerte, 22 con 2, etc.bk_muertes = np.array([0, 1, 2, 3, 4])bk_obs     = np.array([109, 65, 22, 3, 1])n_cuerpos_anio = bk_obs.sum()                          # 109+65+22+3+1total_muertes  = (bk_muertes * bk_obs).sum()           # 0×109 + 1×65 + 2×22 + 3×3 + 4×1bk_lambda      = total_muertes / n_cuerpos_anio         # muertes por cuerpo-añoprint(f"Cuerpos-año observados : {n_cuerpos_anio}")print(f"Muertes en total       : {total_muertes}")print(f"λ = {total_muertes} ÷ {n_cuerpos_anio} = {bk_lambda:.4f} muertes por cuerpo-año")print()# np.repeat reconstruye la lista completa: repite el 0 ciento nueve veces, el 1 sesenta y cinco, etc.# Nos sirve para poder calcular promedio y varianza como con cualquier otro conjunto de datos.bk_datos = np.repeat(bk_muertes, bk_obs)print(f"Promedio  = {bk_datos.mean():.4f}")print(f"Varianza  = {bk_datos.var(ddof=1):.4f}")print(f"Índice de dispersión = {bk_datos.var(ddof=1)/bk_datos.mean():.4f}   <- pegado a 1")

In [ ]:
# Lo que la fórmula de Poisson predice para cada cantidad de muertesbk_esp = stats.poisson.pmf(bk_muertes, bk_lambda) * n_cuerpos_aniotabla_bk = pd.DataFrame({    "muertes en el año": bk_muertes,    "lo que pasó": bk_obs,    "lo que predice Poisson": bk_esp.round(1),    "diferencia": (bk_obs - bk_esp).round(1),})print(tabla_bk.to_string(index=False))fig, ax = plt.subplots()ax.bar(bk_muertes - 0.2, bk_obs, width=0.4, color=AZUL, label="Lo que pasó (1875–1894)")ax.bar(bk_muertes + 0.2, bk_esp, width=0.4, color=ROSA, label=f"Lo que predice Poisson (λ={bk_lambda:.2f})")ax.set_title("Bortkiewicz, 1898: muertes por patada de caballo en un cuerpo, en un año")ax.set_xlabel("muertes"); ax.set_ylabel("cantidad de cuerpos-año"); ax.legend()plt.show()

Mira la columna "diferencia": ninguna pasa de 2 casos. Una fórmula escrita en 1898, sincomputadoras, le pega a la realidad con ese margen.

### 6.2 · Clarke (1946) — las bombas sobre LondresEn 1944 los londinenses estaban convencidos de que las bombas voladoras alemanas caían en**grupos** sobre ciertos barrios, y se mudaban por eso.El actuario R. D. Clarke tomó un mapa del sur de Londres, lo dividió en **576 cuadrículas** (24 ×24) de medio kilómetro por lado, y contó las **537 bombas** registradas.La pregunta: si el patrón resulta Poisson, entonces **nadie estaba apuntando** — es puro azar.

In [ ]:
# Los datos originales de Clarke.# Se leen así: 229 cuadrículas no recibieron ninguna bomba, 211 recibieron una, etc.cl_impactos = np.array([0, 1, 2, 3, 4, 5])     # el 5 agrupa "5 bombas o más"cl_obs      = np.array([229, 211, 93, 35, 7, 1])n_cuadriculas = cl_obs.sum()total_bombas  = 537cl_lambda     = total_bombas / n_cuadriculasprint(f"Cuadrículas  : {n_cuadriculas}")print(f"Bombas       : {total_bombas}")print(f"λ = {total_bombas} ÷ {n_cuadriculas} = {cl_lambda:.4f} bombas por cuadrícula")print()# Para el último grupo ("5 o más") no usamos pmf sino sf, porque hay que sumar# la probabilidad de 5, de 6, de 7... hasta el infinito. sf hace justamente eso.cl_esp = np.append(    stats.poisson.pmf(cl_impactos[:-1], cl_lambda),   # de 0 a 4, uno por uno    stats.poisson.sf(cl_impactos[-2], cl_lambda),     # de 5 en adelante, todo junto) * n_cuadriculastabla_cl = pd.DataFrame({    "bombas en la cuadrícula": ["0", "1", "2", "3", "4", "5 o más"],    "lo que pasó": cl_obs,    "lo que predice el azar": cl_esp.round(1),    "diferencia": (cl_obs - cl_esp).round(1),})print(tabla_cl.to_string(index=False))

In [ ]:
# La prueba formal: ¿la diferencia es demasiado grande para ser casualidad?obs, esp = cl_obs.astype(float).copy(), cl_esp.copy()while len(esp) > 2 and esp[-1] < 5:        # juntamos la cola si tiene menos de 5 esperados    obs[-2] += obs[-1]; esp[-2] += esp[-1]    obs, esp = obs[:-1], esp[:-1]chi2_cl = ((obs - esp) ** 2 / esp).sum()gl_cl   = len(obs) - 1 - 1p_cl    = stats.chi2.sf(chi2_cl, gl_cl)print(f"chi2 = {chi2_cl:.3f}   grados de libertad = {gl_cl}   p = {p_cl:.3f}")print()print("Conclusión:", "el patrón es indistinguible del puro azar" if p_cl > 0.05      else "hay evidencia de que alguien apuntaba")x = np.arange(len(cl_obs))fig, ax = plt.subplots()ax.bar(x - 0.2, cl_obs, width=0.4, color=AZUL, label="Lo que pasó (1944)")ax.bar(x + 0.2, cl_esp, width=0.4, color=ROSA, label=f"Lo que predice el azar (λ={cl_lambda:.2f})")ax.set_xticks(x)                                        # dónde poner las etiquetas del eje Xax.set_xticklabels(["0", "1", "2", "3", "4", "5+"])     # y qué texto poner en cada unaax.set_title("Clarke, 1946: bombas V-1 por cuadrícula en el sur de Londres")ax.set_xlabel("bombas caídas en la cuadrícula"); ax.set_ylabel("cantidad de cuadrículas"); ax.legend()plt.show()

> **Sobre el p-valor.** Clarke reportó 0.88 y aquí sale ≈0.80, porque agrupamos la cola de forma> un poco distinta para cumplir la regla de los 5 casos esperados. La conclusión es la misma.> Guarda este detalle: **el p-valor exacto depende de decisiones que tomas tú**, y por eso hay que> contar cuáles tomaste. Lo retomamos en la Clase 10.> **La lección que hay que llevarse.** El ajuste es casi perfecto: la distribución de las bombas> era **indistinguible del puro azar**. No había barrios apuntados.>> Y hay algo más incómodo. Si le preguntas a cualquiera qué esperaría ver bajo puro azar, casi> siempre contesta "algo parejo". **El azar produce grupitos**, y el ojo humano los lee como> intención. Es exactamente lo que pasa cuando un tablero muestra tres semanas malas seguidas y> alguien propone reorganizar el equipo.>> **Antes de explicar un pico, pregúntate si el azar solo ya lo habría producido.**

### Ejercicio 6.1

In [ ]:
# TODO: aplica la misma prueba formal a los datos de Bortkiewicz.# Ya tienes la función lista del bloque 05: chi2_poisson(datos)# Y ya tienes los datos reconstruidos: bk_datos# Pregunta: ¿la conclusión es la misma que con las bombas de Londres?...

---## 07 · Reto guiado: elegir la distribución correctaPara cada caso: **(a)** di qué distribución es y con qué parámetros, **(b)** explica en uncomentario por qué esa y no otra, **(c)** responde la pregunta con código.El mapa de decisión de la clase:| La pregunta que te hacen | La distribución ||---|---|| ¿Pasó o no pasó? (un solo intento) | Bernoulli(p) || ¿Cuántos "sí" en n intentos fijos? | Binomial(n, p) || ¿Cuántos eventos por hora / por día? | Poisson(λ) || ¿Cuántos intentos hasta el primer "sí"? | Geométrica(p) |Las herramientas de scipy que vas a necesitar: `stats.binom`, `stats.poisson`, `stats.geom`.Todas tienen `.pmf(...)`, `.cdf(...)`, `.sf(...)` y `.ppf(...)`.

In [ ]:
# ══ CASO 1 ═══════════════════════════════════════════════════════════════════# El equipo de ventas manda correos en frío. Responde el 8%.# Pregunta: ¿cuál es la probabilidad de necesitar MÁS de 20 correos#           para conseguir la primera respuesta?## ¿Qué distribución es?  ...# ¿Por qué esa?          ...# Pista: "más de 20 intentos" = sf(20). Y ojo con el ±1.# TODO

In [ ]:
# ══ CASO 2 ═══════════════════════════════════════════════════════════════════# Una pasarela de pagos procesa 15,000 transacciones al día.# La tasa histórica de fraude es del 0.04%.# Pregunta: ¿probabilidad de tener 10 o más transacciones fraudulentas en un día?## Resuélvelo de DOS formas y compara los resultados:#   (a) con la distribución exacta#   (b) con Poisson, usando λ = n × p# ¿Por qué la aproximación funciona aquí? (pista: n es enorme y p es diminuto)# TODO

In [ ]:
# ══ CASO 3 ═══════════════════════════════════════════════════════════════════# Un modelo de fuga de clientes le asigna a un cliente un 23% de probabilidad# de irse este mes.# (a) ¿Qué distribución tiene la variable "este cliente se va"?# (b) De una cartera de 400 clientes con ese mismo 23%, ¿cuántas bajas esperas,#     y entre qué dos números va a estar el 95% de los escenarios posibles?## Pista para (b): stats.binom.ppf(0.025, n, p) y stats.binom.ppf(0.975, n, p)#                 te dan los dos extremos que dejan el 95% en el medio.# TODO

---## Cierre### Tres ideas que no se negocian1. **Nunca ves a toda la población.** Todo lo que reportas es una estimación con un margen de   duda. Y ese margen se puede calcular: es el error estándar.2. **Un promedio solo no describe nada.** Acompáñalo siempre de la mediana o de la dispersión.3. **Elegir una distribución es una decisión tuya**, y se puede defender o refutar con datos.   El índice de dispersión y la prueba de bondad de ajuste son las dos formas de hacerlo.### Reto para la próxima claseElige un número que midan de verdad en tu trabajo (o en tu vida: minutos de espera del bus,mensajes que recibes por hora, lo que sea). Di qué distribución le corresponde y explica en cincolíneas por qué esa y no otra. Si crees que ninguna de las cuatro le calza, explica qué supuesto serompe — esa respuesta también vale.### Clase 2 — Variables continuasQué pasa cuando la variable puede tomar **cualquier** valor y no solo números enteros. Y lapregunta que dejamos abierta hoy: si la probabilidad de un valor exacto es 0, ¿qué es lo queestamos midiendo?---## Bibliografía### Para esta clase, capítulo por capítulo| Tema | Si estás empezando | Si quieres ir más a fondo ||---|---|---|| Promedio, mediana y dispersión | Bruce & Gedeck, cap. 1 | OpenIntro Statistics, cap. 2 || Población, muestra y sesgo | OpenIntro Statistics, cap. 1 | Wasserman, cap. 6 || Variable aleatoria y esperanza | Bruce & Gedeck, cap. 2 | Blitzstein & Hwang, cap. 3 y 4 || Las cuatro distribuciones | Blitzstein & Hwang, cap. 3 | Casella & Berger, cap. 3 || Poisson en la práctica | Bruce & Gedeck, cap. 2 | Downey, *Think Stats* || Por qué esto paga (Módulo 3) | Kohavi & Thomke, HBR 2017 | Kohavi, Tang & Xu, cap. 1 |### Los libros**Empieza por estos dos:**- **Bruce, P., Bruce, A. & Gedeck, P.** (2020). *Practical Statistics for Data Scientists*, 2ª ed.  O'Reilly. Cada concepto con su código en Python. Directo, sin demostraciones.- **Diez, D., Çetinkaya-Rundel, M. & Barr, C.** *OpenIntro Statistics*, 4ª ed. **Gratis** en  openintro.org. El libro más amable que existe para arrancar de cero.**Cuando quieras más:**- **Blitzstein, J. K. & Hwang, J.** (2019). *Introduction to Probability*, 2ª ed. CRC Press.  Enseña cada distribución como una historia. Su curso Stat 110 de Harvard está gratis en YouTube.- **Wasserman, L.** (2004). *All of Statistics*. Springer. Referencia de escritorio: se consulta,  no se lee de corrido. Exige más matemática — no es para empezar.- **Casella, G. & Berger, R. L.** (2002). *Statistical Inference*, 2ª ed. Duxbury. La referencia  formal para cuando necesitas la demostración.- **Kohavi, R., Tang, D. & Xu, Y.** (2020). *Trustworthy Online Controlled Experiments*. Cambridge  University Press. El estándar de la industria en pruebas A/B. Módulo 3.**Gratis y en Python:** Downey, A. — *Think Stats*, 2ª ed. (O'Reilly).### De dónde salen los casos reales- **Daniels, G. S.** (1952). "The 'Average Man'?" *Technical Note WCRD 53-7*, Wright Air Force Base.  Contado también en Rose, T. (2016), *The End of Average*, HarperOne.- **Bortkiewicz, L. von** (1898). *Das Gesetz der kleinen Zahlen*. Leipzig: Teubner.  Reanálisis moderno en Pandit, J. (2016), *Anaesthesia* 71(1).- **Clarke, R. D.** (1946). "An application of the Poisson distribution". *Journal of the Institute  of Actuaries*, 72. Revisado en Shaw & Shaw (2019), *Significance* 16(5).- **Kohavi, R. & Thomke, S.** (2017). "The Surprising Power of Online Experiments".  *Harvard Business Review*, sept–oct 2017.